# Medallion Architecture

A guide to the **Bronze → Silver → Gold** data lakehouse pattern, with real-world implementations:
* **Greenhouse Monitoring** — IoT sensor data → API → multi-level aggregation → Power BI
* **National Nederlanden Savings** — Parquet files → GDPR-compliant Delta → dimensional model
* **Bayer Data Lake** — SAP migration from Data One to Databricks, gold sub-layers
* **Domain-Driven Multi-Catalog** — Unity Catalog per domain, reference vs transactional data

## 📋 What We'll Cover

1. **What is the Medallion Architecture** — the multi-hop pattern and why it exists
2. **Bronze Layer** — raw ingestion, append-only, schema-on-read
3. **Silver Layer** — cleaned, validated, enriched, business-ready
4. **Gold Layer** — aggregated, consumption-ready, optimised for BI
5. **Layer Comparison** — properties, quality, and ownership
6. **Case Study 1: Greenhouse Monitoring** — IoT sensors → API → Bronze → Silver → Gold → Power BI
7. **Case Study 2: National Nederlanden Savings** — Parquet files → GDPR-compliant Delta → Dimensional Model
8. **Case Study 3: Bayer Data Lake** — SAP data, Data One → Databricks, gold sub-layers
9. **Case Study 4: Domain-Driven Multi-Catalog** — one UC catalog per domain, reference vs transactional data

## 1. What is the Medallion Architecture

The **medallion architecture** (also called the **multi-hop architecture**) is a data design pattern that organises a lakehouse into three progressively refined layers:

```
  External Sources                  Lakehouse
  ─────────────────           ──────────────────────────────────────
  APIs                    ┌─────────┐   ┌─────────┐   ┌─────────┐
  Databases       ───▶   │  BRONZE │──▶│  SILVER │──▶│   GOLD  │ ──▶ BI / ML / Apps
  Files / IoT             │   (Raw) │   │ (Clean) │   │  (Agg)  │
  Streaming               └─────────┘   └─────────┘   └─────────┘
```

### Why this pattern?

* **Incremental quality** — each layer improves data quality progressively
* **Replayability** — bronze preserves the original data; you can always reprocess
* **Separation of concerns** — ingestion, business logic, and analytics are decoupled
* **Debuggability** — if gold numbers look wrong, you trace back through silver to bronze
* **Multiple consumers** — different gold tables serve different audiences from the same silver layer

### The medallion is not rigid

It's a **guideline**, not a rule. Some implementations have:
* **Only 2 layers** (bronze + gold) for simple use cases
* **4+ layers** (e.g., a "platinum" layer for ML feature stores)
* **Multiple silver tables** feeding into one gold table
* **Multiple gold tables** from the same silver source (our case study!)

## 2. Bronze Layer — Raw Ingestion

The bronze layer is the **landing zone** for all incoming data. It stores data exactly as it arrived from the source, with minimal or no transformation.

### Principles

| Principle | Detail |
|---|---|
| **Append-only** | New data is inserted; existing rows are never updated or deleted |
| **Schema-on-read** | Store the raw structure; enforce schema only when needed |
| **Full fidelity** | Keep every field from the source, even if you don't need it yet |
| **Idempotent** | Re-running the ingestion should not create duplicates (insert-or-ignore) |
| **Metadata enriched** | Add ingestion timestamp, source system identifier, batch ID |

### Common ingestion patterns

| Pattern | When to use |
|---|---|
| **Auto Loader** (`cloudFiles`) | Files landing in cloud storage (CSV, JSON, Parquet) |
| **API ingestion** | Data pulled from REST APIs on a schedule |
| **Kafka / Event Hubs** | Real-time streaming from message buses |
| **Lakeflow Connect** | Managed connectors to databases (PostgreSQL, Oracle, Salesforce, etc.) |
| **COPY INTO** | One-time or periodic bulk loads from cloud storage |

### Anti-patterns to avoid
* Filtering out "bad" data at bronze — keep everything; filter in silver
* Applying business logic (calculations, joins) — that's silver's job
* Deduplicating based on business keys 

## 3. Silver Layer — Validated & Enriched

The silver layer is where raw data becomes **business-ready**. This is where you apply validation, type casting, deduplication, business calculations, and enrichment.

### Principles

| Principle | Detail |
|---|---|
| **Validated** | Null checks, range checks, referential integrity enforced |
| **Typed** | Strings cast to proper types (timestamps, numerics, enums) |
| **Deduplicated** | Business-key deduplication applied |
| **Enriched** | Derived columns, calculations, and lookups added |
| **Conformed** | Standard naming conventions, units normalised |

### Types of transformations

**1. Data quality (validation)**
```python
# Drop rows missing critical fields
df = df.filter("sensor_id IS NOT NULL AND greenhouse_id IS NOT NULL")

# Validate readings are within physical bounds
df = df.filter("value >= 0 AND value < 10000")
```

**2. Type casting**
```python
# Parse raw string timestamps
df = df.withColumn("recorded_at", to_timestamp(col("recorded_at"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
```

**3. Business calculations**
```python
# Growth rate: (current - previous) / time_delta
# Cumulative growth: running sum since sunrise
# Drain rate: volume drained per unit time
```

**4. Enrichment (joins with reference data)**
```python
# Join with greenhouse dimension for location, timezone, crop type
df = df.join(greenhouse_dim, "greenhouse_id", "left")
```

### What silver should look like

Clean, typed, validated rows with derived business metrics — ready to be aggregated or consumed directly by data-literate users.

## 4. Gold Layer — Aggregated & Consumption-Ready

The gold layer serves **specific business use cases**. It contains pre-aggregated, denormalised, and optimised tables designed for fast querying by BI tools, dashboards, ML models, or applications.

### Principles

| Principle | Detail |
|---|---|
| **Use-case driven** | Each gold table answers a specific business question |
| **Pre-aggregated** | Aggregations computed in advance, not at query time |
| **Denormalised** | Joins pre-computed; no complex joins needed by consumers |
| **Optimised** | Partitioned, clustered, and indexed for fast reads |
| **Multiple tables** | Different aggregation levels serve different needs |

### Common gold patterns

| Pattern | Example |
|---|---|
| **Time-based aggregation** | 5-min, hourly, daily, weekly rollups |
| **Dimensional aggregation** | By region, by customer segment, by product |
| **Fact tables** | Star schema facts for BI tools |
| **Feature tables** | Pre-computed ML features |
| **Reporting snapshots** | Daily/monthly business snapshots |

### Why multiple gold tables?

Different consumers need different granularities:
* **Operations dashboard** needs 5-minute resolution for real-time monitoring
* **Shift managers** need hourly summaries
* **Executives** need daily and weekly trends
* **ML models** need feature tables at prediction-time granularity

Pre-computing these avoids expensive runtime aggregation on large silver tables.

## 5. Layer Comparison

| Property | Bronze | Silver | Gold |
|---|---|---|---|
| **Data quality** | Raw, as-is from source | Validated, cleaned, typed | Aggregated, business-ready |
| **Schema** | Source schema + metadata | Conformed, standardised | Use-case specific |
| **Duplicates** | May contain duplicates | Deduplicated by business key | N/A (pre-aggregated) |
| **Transformations** | None (or minimal) | Filtering, casting, calculations, joins | Aggregation, denormalisation |
| **Update pattern** | Append-only (insert) | Upsert / merge / append | Overwrite or incremental agg |
| **Retention** | Long (months/years) | Medium to long | Medium (refreshed regularly) |
| **Consumers** | Data engineers (debugging) | Data analysts, data scientists | BI tools, executives, apps |
| **Ownership** | Data engineering team | Data engineering / analytics engineering | Analytics / BI team |
| **Volume** | Highest (all raw data) | Slightly less (filtered) | Smallest (aggregated) |
| **Read performance** | Slowest (wide, unoptimised) | Medium | Fastest (pre-computed, clustered) |

### Data flow

```
  Source ──▶ Bronze ──▶ Silver ──▶ Gold ──▶ Consumers
              │            │            │
           Raw data    Validated     Aggregated
           Append      + Enriched    + Optimised
           Full        Business      Use-case
           fidelity    logic         specific
```

## 6. Case Study 1: Greenhouse Monitoring Dashboard

A global agricultural company has **IoT sensors in greenhouses worldwide** measuring growth, weight, drain, temperature, humidity, and light. Data arrives via a **REST API** and needs to be shown in **Power BI** at multiple aggregation levels.

```
 IoT Sensors (global)
       │
       ▼
 External REST API
       │
       ▼
 ┌─────────────────────────────────────────────────────────────────┐
 │  BRONZE              SILVER                   GOLD              │
 │  ┌────────────┐     ┌─────────────────┐     ┌───────────────┐  │
 │  │ Raw API     │     │ Null validation  │     │ agg_5min      │  │
 │  │ INSERT or   │────▶│ Growth rate calc │────▶│ agg_1hour     │  │
 │  │ IGNORE      │     │ Drain calc       │     │ agg_1day      │  │
 │  │             │     │ Growth from      │     │ agg_1week     │  │
 │  │             │     │ sunrise          │     │               │  │
 │  │             │     │ Cumulative growth│     │ (4 tables)    │  │
 │  └────────────┘     └─────────────────┘     └───────┬───────┘  │
 └─────────────────────────────────────────────────────┼──────────┘
                                                       ▼
                                                  POWER BI
                                               (4 dashboard pages)
```

| Layer | What happens |
|---|---|
| **Bronze** | Raw sensor data from API, INSERT or IGNORE (idempotent, append-only) |
| **Silver** | Null checks, growth rate (mm/hr), drain rate (L/hr), growth from sunrise, cumulative growth |
| **Gold** | 4 pre-aggregated tables: 5-min, 1-hour, 1-day, 1-week — each serves a different Power BI page |

## 7. Case Study 2: National Nederlanden — Savings Data Model

National Nederlanden needed a data platform for their **savings product**. Source systems deliver **Parquet files** into Azure Blob Storage. The raw layer was originally in S3, but **GDPR** required deleting customer records — impossible in immutable Parquet files. Solution: move to **Delta tables** which support row-level deletes.

```
 Source Systems (Parquet files)
       │
       ▼
 Azure Blob Storage (landing zone)
       │
       ▼
 ┌─────────────────────────────────────────────────────────────────┐
 │  BRONZE              SILVER                   GOLD              │
 │  ┌────────────┐     ┌─────────────────┐     ┌───────────────┐  │
 │  │ Load Parquet│     │ Deduplication    │     │ Dimension     │  │
 │  │ into Delta  │────▶│ Field quality    │────▶│ tables        │  │
 │  │ tables      │     │ validation       │     │               │  │
 │  │             │     │ Business rules   │     │ Fact tables   │  │
 │  │ Purge source│     │                 │     │               │  │
 │  │ files after │     │                 │     │ (Star Schema) │  │
 │  │ loading     │     │                 │     │               │  │
 │  └────────────┘     └─────────────────┘     └───────┬───────┘  │
 └─────────────────────────────────────────────────────┼──────────┘
                                                       ▼
                                                 BI / Reporting
```

| Layer | What happens |
|---|---|
| **Bronze** | Load Parquet into Delta tables (enables GDPR row-level DELETE), then purge source files |
| **Silver** | Deduplication, field quality validation, business rules |
| **Gold** | Dimensional model — fact tables (transactions, balances) + dimension tables (customer, product, time) |

## 8. Case Study 3: Bayer Data Lake — Migration from Data One to Databricks

Bayer recently migrated from their **Data One** platform to **Databricks** for a lakehouse architecture. The primary data sources are **SAP systems** (ERP, S/4HANA). The gold layer is notable for having **multiple sub-layers** that serve different consumer needs.

```
 SAP Systems (ERP, S/4HANA, BW)
       │
       ▼
 Landing Zone (ADLS / S3)
       │
       ▼
 ┌────────────────────────────────────────────────────────────────────────┐
 │  BRONZE                SILVER                  GOLD                   │
 │  ┌──────────────┐     ┌──────────────┐     ┌──────────────────────┐  │
 │  │ Append raw    │     │ Business      │     │ Sub-layer 1:        │  │
 │  │ SAP extracts  │────▶│ transform-    │────▶│  Raw Expose         │  │
 │  │               │     │ ations        │     │  (SAP data as-is)   │  │
 │  │ Full fidelity │     │               │     │                     │  │
 │  │ No changes    │     │ Conforming    │     │ Sub-layer 2:        │  │
 │  │               │     │ Enrichment    │     │  Active Records     │  │
 │  │               │     │               │     │  (soft deletes      │  │
 │  │               │     │               │     │   filtered out)     │  │
 │  │               │     │               │     │                     │  │
 │  │               │     │               │     │ Sub-layer 3:        │  │
 │  │               │     │               │     │  Data Products      │  │
 │  │               │     │               │     │  (curated, governed │  │
 │  │               │     │               │     │   business models)  │  │
 │  └──────────────┘     └──────────────┘     └──────────────────────┘  │
 └────────────────────────────────────────────────────────────────────────┘
```

| Layer | What happens |
|---|---|
| **Bronze** | Simple append of SAP extracts — full fidelity, no transformation |
| **Silver** | Business transformations — conforming field names, data type standardisation, enrichment |
| **Gold — Raw Expose** | SAP data exposed in its original structure — for consumers who need data exactly as SAP models it |
| **Gold — Active Records** | Soft-deleted records filtered out — only currently active/valid records visible |
| **Gold — Data Products** | Curated, governed business models — purpose-built datasets for specific business domains (e.g., supply chain analytics, R&D reporting) |

### Why multiple gold sub-layers?

Different consumers at Bayer need different views of the same data:
* **SAP specialists** want data in familiar SAP structures → Gold Raw Expose
* **Operational teams** want only active records without deleted/archived rows → Gold Active Records
* **Business analysts** want governed, ready-to-use data products → Gold Data Products

## 9. Case Study 4: Domain-Driven Multi-Catalog Architecture

Large enterprises often use a **shared catalog for bronze and silver** (centralised ingestion and cleansing), then **split into domain-specific catalogs at gold** where each business domain curates its own data products. Reference data lives in its own shared catalog.

```
 ┌──────────────────────────────────────────────────────────────────────────────┐
 │                                                                              │
 │  catalog: enterprise_lake  (shared — central data engineering team)          │
 │  ┌────────────────────────────────────────────────────────────────────────┐  │
 │  │ bronze                              silver                            │  │
 │  │  raw_orders                          orders                           │  │
 │  │  raw_journals                        journals                         │  │
 │  │  raw_shipments                       shipments                        │  │
 │  │  raw_invoices                        invoices                         │  │
 │  │  raw_inventory                       inventory                        │  │
 │  │  raw_customers                       customers                        │  │
 │  │  (all sources land here)             (validated, conformed)            │  │
 │  └────────────────────────────┬───────────────────────────────────────────┘  │
 │                               │                                              │
 │              ┌────────────────┼────────────────┐                             │
 │              ▼                ▼                ▼                              │
 │  catalog: sales      catalog: finance    catalog: supply_chain               │
 │  ┌──────────────┐   ┌──────────────┐   ┌──────────────┐                     │
 │  │ gold         │   │ gold         │   │ gold         │                     │
 │  │  order_facts │   │  gl_facts    │   │  shipping_   │                     │
 │  │  pipeline_   │   │  revenue_    │   │  facts       │                     │
 │  │  metrics     │   │  summary     │   │  stock_levels│                     │
 │  └──────┬───────┘   └──────┬───────┘   └──────┬───────┘                     │
 │         │                  │                  │                              │
 │         └──────────────────┼──────────────────┘                              │
 │                            ▼                                                 │
 │              catalog: reference  (shared dimensions)                         │
 │              ┌─────────────────────────────────────┐                         │
 │              │ gold                                │                         │
 │              │  dim_customer    dim_product         │                         │
 │              │  dim_geography   dim_currency        │                         │
 │              │  dim_date        dim_org_unit        │                         │
 │              └─────────────────────────────────────┘                         │
 └──────────────────────────────────────────────────────────────────────────────┘
```

### How data is organised

| Catalog | Schemas | Ownership | Contains |
|---|---|---|---|
| `enterprise_lake` | `bronze`, `silver` | Central data engineering team | All raw ingestion + validated/conformed data from every source |
| `sales` | `gold` | Sales analytics team | Order facts, pipeline metrics, sales KPIs |
| `finance` | `gold` | Finance analytics team | GL facts, revenue summaries, cost allocation |
| `supply_chain` | `gold` | Supply chain team | Shipping facts, stock levels, demand forecasts |
| `reference` | `gold` | Central platform team | Shared dimensions — customer, product, geography, currency, date |

### Reference (generic) data vs transactional data

| | Reference / Generic Data | Transactional Data |
|---|---|---|
| **What** | Customers, products, geography, currency, dates | Orders, invoices, shipments, journal entries |
| **Bronze/Silver** | In `enterprise_lake` alongside everything else | In `enterprise_lake` alongside everything else |
| **Gold** | Promoted to `reference` catalog (shared read-only) | Split into domain catalogs (`sales`, `finance`, etc.) |
| **Changes** | Slowly changing (SCD Type 2) | Continuously appended / updated |
| **Example join** | `sales.gold.order_facts` JOIN `reference.gold.dim_customer` ON `customer_id` |

### Why shared bronze/silver + domain-specific gold?

* **Single ingestion pipeline** — one team manages all source ingestion, no duplication
* **Consistent cleansing** — validation and conforming happen once in silver, not per domain
* **Domain autonomy at gold** — each team builds their own data products from the shared silver
* **Clear governance** — central team controls what enters the lake; domain teams control what they expose
* **Cross-domain joins** — any domain can read `enterprise_lake.silver.*` for ad-hoc analysis beyond their gold tables